# Structured text and plain-text records - Rust

All 7 Rust examples from [docs/text.md](https://platob.github.io/yggdryl/text/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

## Plain-text records

In [ ]:
use arrow_array::{Array as _, BinaryArray, Int64Array};
use yggdryl::generic::{IORecordOptions as _, RecordOptions};
use yggdryl::io::{Buffer, IOMedia as _};
use yggdryl::text::TextOptions;
use yggdryl::Url;

let text_source = Buffer::from_bytes(
    b"  [INFO] id=7 first  \r\n[WARN] id=9 second\n".to_vec(),
)
.with_media_type(Url::from_str("file:///app.log")?.media_type());

let mut text_options: RecordOptions = TextOptions::new().into();
text_options.set_header(Some(r"\[(?<level>[A-Z]+)\] id=(?<id>\d+)"))?;
text_options.set_lstrip(Some(r"^\s+"))?;
text_options.set_rstrip(Some(r"\s+$"))?;

let text_batch = text_source
    .read_arrow_reader(&text_options)?
    .next()
    .unwrap()?;
assert_eq!(text_batch.schema().fields().len(), 5);
assert_eq!(
    text_batch
        .column(1)
        .as_any()
        .downcast_ref::<Int64Array>()
        .unwrap()
        .values(),
    &[1, 2],
);
assert_eq!(
    text_batch
        .column(2)
        .as_any()
        .downcast_ref::<BinaryArray>()
        .unwrap()
        .value(0),
    b"first",
);

## Raw shared-Scalar access

In [ ]:
use yggdryl::{json, Scalar};

let quote = json::from_utf8(r#"{"symbol":"AAPL","price":12.5}"#)?;

assert_eq!(
    quote.get_key_str("symbol").and_then(Scalar::as_utf8),
    Some("AAPL")
);
assert_eq!(json::into_utf8(&quote)?, r#"{"price":12.5,"symbol":"AAPL"}"#);

### Typed `Scalar` families

In [ ]:
use yggdryl::Scalar;

assert_eq!(
    Scalar::I8(-1).checked_add(&Scalar::U8(2))?,
    Scalar::I16(1),
);
assert_eq!(
    Scalar::d128(1, 0).checked_div(&Scalar::d128(2, 0))?,
    Scalar::d128(5, 1),
);

## Field-directed parsing

In [ ]:
use yggdryl::{json, DataType, Field, Scalar};

let amount = Field::new(
    "amount",
    DataType::decimal128(8, 2)?,
    false,
);
let value = json::from_utf8_with_field(r#""12.50""#, &amount)?;

assert_eq!(value, Scalar::d128(1_250, 2));

## Raw document codecs

In [ ]:
use yggdryl::text::{self, Format};
use yggdryl::Scalar;

let (format, value) = text::from_utf8_inferred(r#"{"id":1}"#)?;

assert_eq!(format, Format::Json);
assert_eq!(value.get_key_str("id"), Some(&Scalar::U64(1)));
assert_eq!(text::into_utf8(&value, format)?, r#"{"id":1}"#);

## Formatting

In [ ]:
use yggdryl::text::Formatting;
use yggdryl::{json, Scalar};

let value = Scalar::from_record([("id", Scalar::I64(1))])?;
let pretty =
    json::into_utf8_with_formatting(&value, Formatting::indented(2))?;

assert_eq!(pretty, "{\n  \"id\": 1\n}");
assert_eq!(json::from_utf8(&pretty)?, value);

## Placeholders

In [ ]:
use yggdryl::text::{Format, Loading, Placeholders};
use yggdryl::Scalar;

let loading = Loading::new().with_placeholders(
    Placeholders::new().with_variable("HOST", Scalar::from("db.internal")),
);
let value = yggdryl::text::from_utf8_with(
    "host: \"{{ HOST }}\"\nport: \"{{ PORT | default(8080) }}\"\n",
    Format::Yaml,
    &loading,
)?;

assert_eq!(value.get_key_str("host").and_then(Scalar::as_utf8), Some("db.internal"));
assert_eq!(value.get_key_str("port"), Some(&Scalar::I64(8080)));